# Paper 3 — E0: UE-Side Adaptive Filtering Baselines

**Experiment E0** (problem statement v0.2): does an online adaptive filter at the UE beat
(i) the trivial predictor $\hat h_t = h_{t-1}$ and (ii) a fixed batch Wiener predictor computed
offline — the WARDEN $W^*$ transferred to the UE? This is the H1 precondition: no margin here means
no intra-cluster localization gain later.

Input: the M2 shakedown datasets (A12, FR1, 10 UEs, 4000 CSI samples at 5 ms), static and dynamic.
No ray tracing — this notebook is pure NumPy and runs in minutes on CPU.

**Predictor.** Each (rx antenna, subcarrier) component of the serving-array CSI is treated as a
scalar complex time series and predicted from its own $p$ past samples: $\hat x_t = w^H u_t$ with
$u_t = [x_{t-1},\dots,x_{t-p}]$. Components are filtered independently but updated in one vectorized
pass. Evaluation is on the second half of each series, so adaptive filters are past their transient.

In [ ]:
# %% [1] Setup (no Sionna needed — NumPy only)
import os, json, uuid, glob, datetime
import numpy as np, h5py
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M2_DIR = os.path.join(BASE, 'm2')
E0_DIR = os.path.join(BASE, 'e0'); os.makedirs(E0_DIR, exist_ok=True)

def drive_canary(path=E0_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [2] Configuration
ORDERS = (1, 2, 4, 8)           # filter length p
MU_LMS = 0.01                   # LMS step (on energy-normalized data)
MU_NLMS = 0.1                   # NLMS step in (0, 2); smaller = less misadjustment, slower tracking.
                                # Swept in cell [5b] because the best value differs by regime.
RLS_LAMBDA = 0.995              # forgetting factor
RLS_DELTA = 1e-2                # P(0) = delta^-1 I
RCOND = 1e-4                    # singular values below RCOND*max are dropped in the Wiener solve.
                                # Calibrated in cell [6b]: weaker truncation KEEPS noise-dominated
                                # modes and degrades even the oracle (0.0120 vs 0.0012 at p=4 static),
                                # i.e. the CSI series has low effective rank — the thesis H5 thread.
TRAIN_FRAC = 0.5                # first half: batch-Wiener fitting + adaptive transient
SEED = 2026
np.random.seed(SEED)

In [ ]:
# %% [3] Load M2 datasets (new Doppler format only) and build serving-link series
def load_series(h5path):
    with h5py.File(h5path, 'r') as f:
        if 'fine_dt' not in f.attrs or f['csi'].ndim != 5:
            return None
        H = f['csi'][:]                                   # [T, U, rx_ant, A, S]
        meta = dict(regime=str(f.attrs['regime']), n_scat=int(f.attrs['n_scatterers']),
                    dt=float(f.attrs['fine_dt']))
    E = (np.abs(H)**2).sum(axis=(2, 4))                   # [T, U, A]
    serve = E.mean(0).argmax(1)                           # serving array per UE
    U = H.shape[1]
    X = np.stack([H[:, u, :, serve[u], :].reshape(H.shape[0], -1) for u in range(U)], axis=1)
    # X: [T, U, C] with C = rx_ant * S ; normalize each link to unit mean power (scale-free NMSE)
    X = X / (np.sqrt((np.abs(X)**2).mean(axis=(0, 2), keepdims=True)) + 1e-30)
    return X.astype(np.complex64), meta, serve

DATA = {}
for p_ in sorted(glob.glob(os.path.join(M2_DIR, 'paper3_M2_*.h5'))):
    out = load_series(p_)
    if out is None:
        print('SKIPPED (old format):', os.path.basename(p_)); continue
    X, meta, serve = out
    DATA[meta['regime']] = (X, meta)
    print(f"{meta['regime']:8s} | X {X.shape} [T, U, C] | dt {meta['dt']*1000:.0f} ms | "
          f"N_s {meta['n_scat']} | serving arrays {serve.tolist()}")
assert DATA, 'no usable M2 datasets found'

In [ ]:
# %% [4] Predictors: naive, batch Wiener (fixed W*), LMS, NLMS, RLS
# All operate on X [T, K] where K = U*C independent complex components, updated in one vectorized pass.

def lag_matrix(X, p, t):
    """u_t = [x_{t-1}, ..., x_{t-p}] as [p, K]."""
    return X[t-p:t][::-1]

def nmse(err, ref, mask):
    return float((np.abs(err[mask])**2).sum() / ((np.abs(ref[mask])**2).sum() + 1e-30))

def run_naive(X, p):
    e = X[p:] - X[p-1:-1]
    return e, X[p:]

def run_wiener(X, p, t_split, fit='train'):
    """Batch Wiener-Hopf per component, fitted on the training half, applied throughout.
    This is WARDEN's W* transferred to the UE: optimal but fixed."""
    K = X.shape[1]
    r = np.zeros((p+1, K), np.complex128)
    Xtr = X[:t_split] if fit == 'train' else X[t_split:]
    for k in range(p+1):
        r[k] = (Xtr[k:] * np.conj(Xtr[:len(Xtr)-k or None])).mean(0) if k else (np.abs(Xtr)**2).mean(0)
    # predictor y = w^H u with u_i = x_{t-1-i}:  R w = q,  R[i,j] = E[u_i u_j*] = r(j-i),
    # q_i = E[u_i d*] = conj(r(i+1)).  (r(-k) = conj(r(k)).)
    R = np.zeros((K, p, p), np.complex128)
    for i in range(p):
        for j in range(p):
            R[:, i, j] = np.conj(r[i-j]) if i >= j else r[j-i]
    # Oversampled CSI makes R nearly singular (adjacent samples almost identical), so loading is
    # scaled to the signal power, not absolute: with a fixed 1e-6 the p>=4 solutions blew up.
    # CSI at 5 ms is correlated to ~0.999, so R is near-singular: diagonal loading still let the
    # p>=4 solutions blow up (NMSE 85 at p=8), because a huge-norm w fitted on one half does not
    # transfer to the other. Truncating the degenerate modes with a pseudo-inverse caps ||w||.
    q = np.conj(r[1:p+1]).T[..., None]
    w = (np.linalg.pinv(R, rcond=RCOND) @ q)[..., 0].T               # [p, K]
    e, d = [], []
    for t in range(p, len(X)):
        u = lag_matrix(X, p, t)
        e.append(X[t] - (np.conj(w)*u).sum(0)); d.append(X[t])
    return np.array(e), np.array(d)

def run_adaptive(X, p, kind):
    K = X.shape[1]
    w = np.zeros((p, K), np.complex128)
    # LMS step scaled by the component's average power (measured offline): still a FIXED step per
    # component, unlike NLMS which renormalizes per sample. Without it a power transient diverges.
    mu_lms = MU_LMS/(p*(np.abs(X)**2).mean(0) + 1e-9)
    if kind == 'rls':
        P = np.tile((1.0/RLS_DELTA)*np.eye(p, dtype=np.complex128), (K, 1, 1))
    e_all, d_all = [], []
    for t in range(p, len(X)):
        u = lag_matrix(X, p, t).astype(np.complex128)
        yhat = (np.conj(w)*u).sum(0)
        e = X[t] - yhat
        e_all.append(e); d_all.append(X[t])
        if kind == 'lms':
            w += mu_lms * u * np.conj(e)
        elif kind == 'nlms':
            w += MU_NLMS * u * np.conj(e) / ((np.abs(u)**2).sum(0) + 1e-6)
        else:
            Pu = np.einsum('kij,jk->ik', P, u)
            den = RLS_LAMBDA + (np.conj(u)*Pu).sum(0)
            g = Pu/den
            w += g*np.conj(e)
            uHP = np.einsum('ik,kij->jk', np.conj(u), P)
            P = (P - np.einsum('ik,jk->kij', g, uHP))/RLS_LAMBDA
    return np.array(e_all), np.array(d_all)

In [ ]:
# %% [5] Sweep: regimes x algorithms x filter orders
import time
rows, curves = [], {}
for regime, (X, meta) in DATA.items():
    T, U, C = X.shape
    Xf = X.reshape(T, U*C)                                  # independent components
    t_split = int(TRAIN_FRAC*T)
    for p in ORDERS:
        for algo in ('naive', 'wiener', 'wiener_oracle', 'lms', 'nlms', 'rls'):
            if algo == 'naive' and p != ORDERS[0]:
                continue                                    # order-independent
            t0 = time.time()
            if algo == 'naive':   e, d = run_naive(Xf, p)
            elif algo == 'wiener': e, d = run_wiener(Xf, p, t_split, fit='train')
            elif algo == 'wiener_oracle': e, d = run_wiener(Xf, p, t_split, fit='test')
            else:                  e, d = run_adaptive(Xf, p, algo)
            mask = np.zeros(len(e), bool); mask[t_split-p:] = True      # evaluate post-transient
            val = nmse(e, d, mask)
            # per-UE spread
            eu = e.reshape(len(e), U, C)[mask]; du = d.reshape(len(d), U, C)[mask]
            per_ue = ((np.abs(eu)**2).sum(axis=(0, 2))/((np.abs(du)**2).sum(axis=(0, 2))+1e-30))
            rows.append(dict(regime=regime, algo=algo, p=p, nmse=val,
                             nmse_p10=float(np.percentile(per_ue, 10)),
                             nmse_p90=float(np.percentile(per_ue, 90)), secs=time.time()-t0))
            if p == 4 or algo == 'naive':
                win = 50
                inst = (np.abs(e)**2).sum(1)/((np.abs(d)**2).sum(1)+1e-30)
                curves[f'{regime}/{algo}'] = np.convolve(inst, np.ones(win)/win, 'valid')
            print(f"{regime:8s} {algo:6s} p={p} | NMSE {val:.4f} | {time.time()-t0:5.1f} s")

In [ ]:
# %% [5b] Step-size / forgetting-factor sensitivity (p = 4, both regimes)
# Adaptive filters trade misadjustment against tracking speed; the static and dynamic regimes
# generally want different settings, and that difference is itself an H2 signal.
sens = []
for regime, (X, meta) in DATA.items():
    T, U, C = X.shape; Xf = X.reshape(T, U*C); t_split = int(TRAIN_FRAC*T)
    mask = None
    for mu in (1.0, 0.5, 0.2, 0.1, 0.05):
        globals()['MU_NLMS'] = mu
        e, d = run_adaptive(Xf, 4, 'nlms')
        mask = np.zeros(len(e), bool); mask[t_split-4:] = True
        sens.append(dict(regime=regime, knob='nlms_mu', value=mu, nmse=nmse(e, d, mask)))
    globals()['MU_NLMS'] = 0.1
    for lam in (0.9995, 0.999, 0.997, 0.995, 0.99, 0.98):
        globals()['RLS_LAMBDA'] = lam
        e, d = run_adaptive(Xf, 4, 'rls')
        mask = np.zeros(len(e), bool); mask[t_split-4:] = True
        sens.append(dict(regime=regime, knob='rls_lambda', value=lam, nmse=nmse(e, d, mask)))
    globals()['RLS_LAMBDA'] = 0.995
for s in sens:
    print(f"{s['regime']:8s} {s['knob']:11s} {s['value']:<6} NMSE {s['nmse']:.4f}")

In [ ]:
# %% [6b] Diagnostic: pinv truncation sensitivity of the oracle Wiener
# Ran once to calibrate RCOND. Kept for the record: NMSE worsens as rcond shrinks, so the
# weakly-determined correlation modes carry estimation noise, not signal (low effective rank).
if False:                                                   # flip to True to re-run the calibration
    for regime, (X, meta) in DATA.items():
        T, U, C = X.shape; Xf = X.reshape(T, U*C); t_split = int(TRAIN_FRAC*T)
        for p in (2, 4, 8):
            line = f'{regime:8s} p={p} |'
            for rc in (1e-4, 1e-6, 1e-8, 1e-10):
                globals()['RCOND'] = rc
                e, d = run_wiener(Xf, p, t_split, fit='test')
                m = np.zeros(len(e), bool); m[t_split-p:] = True
                line += f'  rcond {rc:.0e}: {nmse(e, d, m):.4f}'
            print(line)
    globals()['RCOND'] = 1e-4

In [ ]:
# %% [6] Results table and H1 check
import pandas as pd
df = pd.DataFrame(rows)
piv = df.pivot_table(index=['regime', 'algo'], columns='p', values='nmse')
print(piv.round(4).to_string())

print()
for regime in DATA:
    base = df[(df.regime == regime) & (df.algo == 'naive')].nmse.iloc[0]
    best = df[(df.regime == regime) & (df.algo != 'naive')].nmse.min()
    brow = df[(df.regime == regime) & (df.nmse == best)].iloc[0]
    wien = df[(df.regime == regime) & (df.algo == 'wiener')].nmse.min()
    ad = df[(df.regime == regime) & (df.algo.isin(['lms', 'nlms', 'rls']))].nmse.min()
    orc = df[(df.regime == regime) & (df.algo == 'wiener_oracle')].nmse.min()
    print(f'{regime:8s} | naive {base:.4f} | best fixed Wiener {wien:.4f} | best adaptive {ad:.4f} '
          f'| best overall {brow.algo} p={brow.p} ({(1-best/base)*100:.1f}% below naive)')
    print(f'{"":8s} | [H1a] adaptive beats naive by >20%: '
          f'{"PASS" if ad < 0.8*base else "FAIL"}'
          f' | [H1b] adaptive beats fixed Wiener: {"PASS" if ad < wien else "FAIL"}')
    print(f'{"":8s} | [H1b-strict] adaptive beats the ORACLE fixed Wiener ({orc:.4f}): '
          f'{"PASS" if ad < orc else "FAIL"}')

In [ ]:
# %% [7] Learning curves (p = 4) and per-UE spread
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, c in sorted(curves.items()):
    t_ms = np.arange(len(c))*5
    ax[0].plot(t_ms, c, lw=1, label=name)
ax[0].set_xlabel('time [ms]'); ax[0].set_ylabel('instantaneous NMSE (50-sample MA)')
ax[0].set_yscale('log'); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=7)
ax[0].set_title('Convergence, p = 4')

sub = df[df.algo != 'naive']
for i, (regime, g) in enumerate(sub.groupby('regime')):
    for algo, gg in g.groupby('algo'):
        ax[1].errorbar(gg.p + 0.1*i, gg.nmse,
                       yerr=[gg.nmse-gg.nmse_p10, gg.nmse_p90-gg.nmse], marker='o',
                       capsize=3, lw=1, label=f'{regime}/{algo}')
for regime, g in df[df.algo == 'naive'].groupby('regime'):
    ax[1].axhline(g.nmse.iloc[0], ls='--', lw=1, color='gray')
ax[1].set_xlabel('filter order p'); ax[1].set_ylabel('NMSE (P10-P90 across UEs)')
ax[1].set_xscale('log', base=2); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=7)
ax[1].set_title('NMSE vs order (dashed: naive baselines)')
plt.tight_layout(); plt.savefig(os.path.join(E0_DIR, 'e0_results.png'), dpi=150); plt.show()

In [ ]:
# %% [8] Complexity accounting (input to E2)
# Complex MACs per update per component. 1 complex MAC = 4 real MACs + 2 adds.
for p in ORDERS:
    print(f'p={p:2d} | predict {p:3d} | LMS +{p:3d} | NLMS +{2*p:3d} | RLS +{3*p*p + 2*p:4d} cMAC/component/sample')
print('\nPer CSI sample the UE filters C = rx_ant * S components; the E2 energy model multiplies')
print('these counts by C, by the sample rate, and by published per-MAC energy for mobile SoCs.')

In [ ]:
# %% [9] Save results + backup
drive_canary()
out_json = os.path.join(E0_DIR, f'e0_results_{datetime.datetime.now():%Y%m%d_%H%M}.json')
with open(out_json, 'w') as f:
    json.dump(dict(rows=rows, config=dict(orders=list(ORDERS), mu_lms=MU_LMS, mu_nlms=MU_NLMS,
                                          rls_lambda=RLS_LAMBDA, train_frac=TRAIN_FRAC, seed=SEED)),
              f, indent=2)
df.to_csv(os.path.join(E0_DIR, 'e0_results.csv'), index=False)
print('Saved:', out_json)

import shutil
zip_path = shutil.make_archive('/content/paper3_e0_backup', 'zip', E0_DIR)
from google.colab import files as colab_files
colab_files.download(zip_path)